# 5. Reaction reports

The two questions a sensitivity study is meant to answer, as tables:
1. **Before any sweep** — which reactions actually mattered in the baseline
   (`flux_reaction_list`)?
2. **After a sweep** — which of those turned out sensitive enough to be
   worth a better rate (`sensitivity_reaction_report`)?

Together these are the "comprehensive reaction list" a sensitivity study
should hand off: experimentalists want to know what to remeasure,
astrophysicists want to know what rate (and what uncertainty) to put in the
next hydro simulation.

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using NuGridJl

NUPPN = joinpath(@__DIR__, "..", "test", "data", "nuppn_data")
SWEEP_DIR = joinpath(@__DIR__, "..", "test", "data", "sweep")

run = PPNRun(NUPPN)
sweep = PPNSweep(SWEEP_DIR)

  Activating 

project at `~/Documents/NovaNucleosynthesis/NuGrid-Tools/NuGridJl`


PPNSweep("/home/sgervais/Documents/NovaNucleosynthesis/NuGrid-Tools/NuGridJl/demos/../test/data/sweep", 1 reactions, 2 factored runs)

## Pre-study: what carried flux in the baseline

`flux_reaction_list(run; cycle, threshold)` — every reaction above
`threshold` at that cycle, ranked by flux, with its source and printed rate.
This is your starting point for deciding what to even put in a reaction
plan for a sweep.

In [2]:
flist = flux_reaction_list(run; cycle = :final, threshold = 1e-30)
flist[1:10, :]

Row,index,reaction,source,rtype,active,flux,rate
,Int64,String,String,String,Bool,Float64,Float64
1,161,"13C(p,g)14N",NACRR,"(p,g)",true,4.29619e-6,2.335e-21
2,181,"13N(+,g)13C",NETB1,"(+,g)",true,3.86147e-6,0.00116
3,154,"12C(p,g)13N",NACRR,"(p,g)",true,1.41068e-6,5.63e-22
4,227,"15O(+,g)15N",NETB1,"(+,g)",true,1.33556e-6,0.005691
5,201,"15N(p,a)12C",NACRR,"(p,a)",true,1.33411e-6,1.192e-20
6,17,"14N(p,g)15O",VITAL,"(p,g)",true,1.19944e-6,3.28e-25
7,245,"17O(p,a)14N",NACRR,"(p,a)",true,4.7925e-8,1.0e-99
8,215,"14O(+,g)14N",NETB1,"(+,g)",true,1.79267e-8,0.009815
9,16,"13N(p,g)14O",VITAL,"(p,g)",true,1.59383e-8,5.162e-25


## Post-study: what turned out sensitive

`sensitivity_reaction_report(table; threshold)` collapses a long-format
`sensitivity_table` (notebook 4) to one row per reaction: how many
isotopes/factors it was tested against, its biggest observed swing, and
whether that crosses `threshold` — the ranked "what to remeasure" list.

In [3]:
table = sensitivity_table(sweep, "He-4")
report = sensitivity_reaction_report(table)

Row,reaction,n_factors,n_isotopes,max_abs_log_ratio,worst_isotope,worst_factor,sensitive
,String,Int64,Int64,Float64,String,Float64,Bool
1,13N_pg_14O,2,1,0.197833,He-4,0.5,true


## Exporting

Every `DataFrame` in `NuGridJl` — these reports included — can go straight
to Markdown, HTML, or CSV.

In [4]:
print(dataframe_to_markdown(report))

| reaction | n_factors | n_isotopes | max_abs_log_ratio | worst_isotope | worst_factor | sensitive |
| --- | --- | --- | --- | --- | --- | --- |
| 13N_pg_14O | 2 | 1 | 0.1978 | He-4 | 0.5 | true |

In [5]:
dataframe_to_html(report)

reaction,n_factors,n_isotopes,max_abs_log_ratio,worst_isotope,worst_factor,sensitive
13N_pg_14O,2,1,0.1978,He-4,0.5,true


In [6]:
mktempdir() do dir
    path = joinpath(dir, "sensitivity_report.csv")
    save_table(report, path)
    isfile(path), filesize(path)
end

(true, 134)

## Where this goes next

Point `flux_reaction_list`/`sensitivity_reaction_report` at a real sweep
(built with `tools/build_sweep.jl`, see its `README.md`) instead of this
notebook's tiny fixture, and these two tables are the actual deliverable: a
pre-study flux-ranked reaction list, and a post-study sensitivity-ranked
one, both exportable straight into a paper or a hydro-sim rate table.

That's the last of the currently-implemented demos. Monte Carlo ensemble
analysis (sampling STARLIB's real per-reaction, per-temperature uncertainty
rather than a single deterministic factor) is designed but not yet
implemented — coming alongside `tools/build_ensemble.jl`.